# SoundTag AST Genre Classifier (Kaggle)

Audio Spectrogram Transformer fine-tuning for K-pop genre classification.

**Settings → Accelerator → GPU P100**

In [ ]:
# 1. 설치
!pip install -q transformers datasets accelerate evaluate torchaudio

In [ ]:
# 2. GPU 확인
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# 3. 데이터 확인
# Kaggle Dataset: /kaggle/input/soundtag-genre-previews/
import os
from pathlib import Path

DATASET_DIR = '/kaggle/input/soundtag-genre-previews'

# zip 안의 구조 확인
for root, dirs, files in os.walk(DATASET_DIR):
    level = root.replace(DATASET_DIR, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 2:
        for d in sorted(dirs):
            subfiles = len(list(Path(root, d).rglob('*.mp3')))
            print(f'{indent}  {d}/ ({subfiles} mp3s)')
    if level >= 2:
        print(f'{indent}  [{len(files)} files]')
        break

In [ ]:
# 4. 프리뷰 디렉토리 찾기
# zip 구조에 따라 경로가 다를 수 있음
PREVIEW_DIR = None
for candidate in [
    os.path.join(DATASET_DIR, 'genre_data', 'previews'),
    os.path.join(DATASET_DIR, 'previews'),
    DATASET_DIR,
]:
    if os.path.exists(candidate):
        subdirs = [d for d in os.listdir(candidate) if os.path.isdir(os.path.join(candidate, d))]
        if any('Pop' in d or 'Trap' in d or 'EDM' in d or 'Hip' in d for d in subdirs):
            PREVIEW_DIR = candidate
            break

if PREVIEW_DIR:
    genres = sorted([d for d in os.listdir(PREVIEW_DIR) if os.path.isdir(os.path.join(PREVIEW_DIR, d))])
    total = 0
    for g in genres:
        count = len(list(Path(PREVIEW_DIR, g).glob('*.mp3')))
        total += count
        print(f'  {g}: {count}')
    print(f'\nTotal: {total} files across {len(genres)} genres')
    print(f'Preview dir: {PREVIEW_DIR}')
else:
    print('❌ Preview directory not found!')
    print('Dataset contents:')
    !find /kaggle/input/soundtag-genre-previews -maxdepth 3 -type d | head -30

In [ ]:
# 5. 데이터셋 구성
import torchaudio
import numpy as np
from torch.utils.data import Dataset
from transformers import ASTFeatureExtractor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

class GenreDataset(Dataset):
    def __init__(self, file_paths, labels, feature_extractor, target_sr=16000):
        self.file_paths = file_paths
        self.labels = labels
        self.feature_extractor = feature_extractor
        self.target_sr = target_sr

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        path = self.file_paths[idx]
        label = self.labels[idx]

        try:
            waveform, sr = torchaudio.load(path)
            if waveform.shape[0] > 1:
                waveform = waveform.mean(dim=0, keepdim=True)
            waveform = waveform.squeeze(0)

            if sr != self.target_sr:
                resampler = torchaudio.transforms.Resample(sr, self.target_sr)
                waveform = resampler(waveform)

            inputs = self.feature_extractor(
                waveform.numpy(),
                sampling_rate=self.target_sr,
                return_tensors='pt'
            )
            input_values = inputs['input_values'].squeeze(0)

        except Exception as e:
            print(f'Error loading {path}: {e}')
            input_values = torch.zeros(1024, 128)

        return {'input_values': input_values, 'labels': label}


# 파일 수집
file_paths = []
genre_labels = []

for genre_dir in sorted(Path(PREVIEW_DIR).iterdir()):
    if genre_dir.is_dir():
        genre_name = genre_dir.name.replace('_', ' ').replace('RandB', 'R&B')
        for mp3 in genre_dir.glob('*.mp3'):
            file_paths.append(str(mp3))
            genre_labels.append(genre_name)

print(f'Total files: {len(file_paths)}')
print(f'Genres: {len(set(genre_labels))}')

# 레이블 인코딩
le = LabelEncoder()
encoded_labels = le.fit_transform(genre_labels)
num_labels = len(le.classes_)
print(f'Classes: {list(le.classes_)}')

# Train/test split
train_paths, test_paths, train_labels, test_labels = train_test_split(
    file_paths, encoded_labels, test_size=0.2, random_state=42, stratify=encoded_labels
)
print(f'Train: {len(train_paths)}, Test: {len(test_paths)}')

In [ ]:
# 6. Feature extractor + 데이터셋 정규화
feature_extractor = ASTFeatureExtractor.from_pretrained(
    'MIT/ast-finetuned-audioset-10-10-0.4593'
)

# 샘플로 mean/std 추정
sample_indices = np.random.choice(len(train_paths), min(100, len(train_paths)), replace=False)
all_specs = []

for idx in sample_indices:
    try:
        waveform, sr = torchaudio.load(train_paths[idx])
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        waveform = waveform.squeeze(0)
        if sr != 16000:
            resampler = torchaudio.transforms.Resample(sr, 16000)
            waveform = resampler(waveform)
        inputs = feature_extractor(waveform.numpy(), sampling_rate=16000, return_tensors='pt')
        all_specs.append(inputs['input_values'].squeeze(0).numpy())
    except:
        pass

all_specs_flat = np.concatenate([s.flatten() for s in all_specs])
dataset_mean = float(np.mean(all_specs_flat))
dataset_std = float(np.std(all_specs_flat))
print(f'Dataset mean: {dataset_mean:.4f}')
print(f'Dataset std:  {dataset_std:.4f}')

feature_extractor.mean = dataset_mean
feature_extractor.std = dataset_std

In [ ]:
# 7. 데이터셋 생성 + 샘플 확인
train_dataset = GenreDataset(train_paths, train_labels, feature_extractor)
test_dataset = GenreDataset(test_paths, test_labels, feature_extractor)

sample = train_dataset[0]
print(f'Input shape: {sample["input_values"].shape}')
print(f'Label: {sample["labels"]} ({le.classes_[sample["labels"]]})')

In [ ]:
# 8. AST 모델 로드
from transformers import ASTForAudioClassification

model = ASTForAudioClassification.from_pretrained(
    'MIT/ast-finetuned-audioset-10-10-0.4593',
    num_labels=num_labels,
    ignore_mismatched_sizes=True,
)

total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total_params:,}')
print(f'Trainable:    {trainable:,}')
print(f'Labels:       {num_labels}')

In [ ]:
# 9. 학습 설정
from transformers import TrainingArguments, Trainer
import evaluate

accuracy_metric = evaluate.load('accuracy')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    
    # Top-3 accuracy
    top3 = 0
    for i in range(len(labels)):
        top3_preds = np.argsort(logits[i])[-3:]
        if labels[i] in top3_preds:
            top3 += 1
    acc['top3_accuracy'] = top3 / len(labels)
    
    return acc

training_args = TrainingArguments(
    output_dir='/kaggle/working/ast_genre_model',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=1e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=15,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    save_total_limit=2,
    fp16=True,
    dataloader_num_workers=2,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

print('✅ Trainer ready')

In [ ]:
# 10. 학습!
trainer.train()

In [ ]:
# 11. 최종 평가
results = trainer.evaluate()
print(f"\n{'='*50}")
print(f"📊 Final Results:")
print(f"   Accuracy:       {results['eval_accuracy']:.1%}")
print(f"   Top-3 Accuracy: {results['eval_top3_accuracy']:.1%}")
print(f"{'='*50}")

In [ ]:
# 12. 상세 분류 리포트
from sklearn.metrics import classification_report

predictions = trainer.predict(test_dataset)
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = predictions.label_ids

print(classification_report(y_true, y_pred, target_names=le.classes_, zero_division=0))

In [ ]:
# 13. 모델 저장
import pickle

save_path = '/kaggle/working/ast_genre_model_final'
trainer.save_model(save_path)
feature_extractor.save_pretrained(save_path)

with open(os.path.join(save_path, 'label_encoder.pkl'), 'wb') as f:
    pickle.dump(le, f)

print(f'💾 Model saved to: {save_path}')

# zip으로 압축 (다운로드용)
import shutil
shutil.make_archive('/kaggle/working/ast_genre_model_final', 'zip', save_path)
print('💾 Zipped: /kaggle/working/ast_genre_model_final.zip')